# DAT09 · 미션10 전체 실행을 검증하고 담당자에게 인계하기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션9을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m09.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m10_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=9):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 9)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션10 전체 실행을 검증하고 담당자에게 인계하기

예상 3시간입니다.

#### 이번 미션의 목표

새 프로세스에서 전체 도구를 재실행하고, 원자료부터 화면까지 값을 추적해 다른 담당자가 이어서 사용할 수 있게 인계합니다.

### 1단계 빈 결과 폴더에서 전체를 한 명령으로 실행하기

#### 이 단계에서 하는 일

현재 노트북에 남아 있는 변수나 예전 결과 파일에 의존하면 다른 담당자는 같은 결과를 만들 수 없습니다. 저장된 코드와 입력만으로 새 Python 프로세스에서 처음부터 실행되는지 확인합니다.

#### 시작 전 확인

미션9까지 완료한 결과를 사용합니다. 노트북의 m10_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션1~미션9와 현재 작성한 m10 함수, 저장된 src/m01.py~m10.py, data·config·공통 코드를 사용합니다. 새 환경은 새 Python 프로세스와 빈 결과 폴더를 뜻하며 새 OS 설치를 뜻하지 않습니다.

#### 작업 순서

- run_dat09.py가 src의 단계 함수를 읽어 미션1부터 미션10까지 순서대로 실행하게 하세요. 수집·변환·DB·동기화·품질·후보·화면·시험이 실제로 이어져야 합니다.

- --output 인자로 매번 새 결과 폴더를 받게 하고, 부모 프로세스의 c를 전달하지 말고 context를 새로 만드세요.

- 명령 `python run_dat09.py --output 새_폴더`를 새 프로세스로 실행하세요. 미션10 안에서 다시 새 프로세스를 무한히 만드는 일이 없도록 자식 실행에서는 재귀 검증을 생략합니다.

- 종료 코드, 전체 로그와 완료 미션 목록을 저장하세요. 실패하면 마지막 성공 미션·예외·이유를 run_failure.json에 남기고 완료로 표시하지 않습니다.


#### 저장할 결과

작업 루트의 run_dat09.py, clean_run.log와 새 폴더의 run_summary.json 또는 run_failure.json을 남깁니다.

#### 직접 확인하는 방법

종료 코드 0, 완료 목록 1~10, 새 결과 폴더의 열 미션 결과를 확인하세요. 이전 폴더의 파일이 있다는 이유로 성공 처리하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 다섯 종류의 값을 원파일부터 화면까지 추적하기

#### 이 단계에서 하는 일

최종 차트 값이 맞는지 알려면 원래 값이 어떤 변환과 판정을 거쳤는지 확인해야 합니다. 정상뿐 아니라 단위·결측·충돌·지연도 같은 방식으로 따라가야 합니다.

#### 시작 전 확인

미션10의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

원파일, 미션2 normalized.csv, 미션3 DB·logical_stock.csv, 미션4 후보, 미션7 승인 상태와 미션6 화면을 사용합니다. 대표 ID는 정상 A-0719, BOX 환산 B-0719, 결측 A-0000, 값 충돌 B-0030, 지연 C-0060입니다.

#### 작업 순서

- 다섯 ID를 각각 원파일에서 찾고 원래 수량·단위·시각을 기록하세요. 변환 후 EA·UTC·코드와 검사 상태를 같은 행에 적습니다.

- DB에 저장된 원래 보고가 파일의 내용과 같은지 확인하고, 해당 재고 이력에 연결된 세 출처 ID를 찾으세요.

- 후보 유무·후보 수량·승인 상태와 화면 수량·검토 상태를 비교하세요. 결측의 후보를 원래 수량처럼 적거나, 지연으로 제외된 수량을 0으로 쓰지 않습니다.

- 각 사례에 사용한 코드 매핑·검증 규칙 버전과 대조 결과를 남기고 integrated_dashboard.html에서도 같은 대상을 확인하세요.


#### 저장할 결과

lineage_samples.csv와 integrated_dashboard.html, 다섯 사례의 후보·승인 대조 기록을 남깁니다.

#### 직접 확인하는 방법

다섯 ID가 모두 원파일·변환 보고·DB·화면에 연결돼야 합니다. 기본 실행의 승인 상태는 UNAPPROVED이고 확인 불가 화면값은 N/A입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 승인 없는 실행과 세 장애 상황 확인하기

#### 이 단계에서 하는 일

전체 실행을 편하게 만들려고 승인을 자동으로 통과시키거나 장애를 숨기면 인수할 수 없습니다. 기본 실행은 후보를 보류한 상태로 끝나야 하며 장애 때 원자료와 정상 결과가 어떻게 보호되는지 확인해야 합니다.

#### 시작 전 확인

미션10의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

기본 DB·후보 목록, load_fn, 미션4 checkpoint.json과 원파일 해시를 사용합니다. 파일 누락·DB 잠김은 별도 시험 경로에서 만들고 실제 기본 파일을 삭제하지 않습니다.

#### 작업 순서

- 기본 승인 마스터가 0건이고 모든 외부 반영이 NOT_CONNECTED인지 조회하세요. 보류·후보 목록이 그대로 남는지 확인합니다.

- 없는 파일 경로를 읽어 수집 실패가 나는지 확인하고 기존 원파일 해시를 비교하세요. 실패한 단계를 성공으로 기록하지 않습니다.

- 임시 DB를 두 연결로 열어 한쪽이 잠근 동안 다른 쪽의 저장 실패를 확인하세요. 재시도가 필요하다는 처리와 기록을 남깁니다. 미션8의 실제 재시도 시험도 함께 제시합니다.

- 체크포인트에서 복원한 처리기에 전체 이벤트를 다시 보내고 재수신 전후 현재값·충돌 목록이 같은지 비교하세요. 시험 범위를 넘는 복구를 했다고 주장하지 않습니다.


#### 저장할 결과

pending_cases.csv, e2e_tests.json과 기본 승인 건수·원파일 해시·장애 로그를 남깁니다.

#### 직접 확인하는 방법

기본 승인 0건, 세 장애 시험 통과를 확인하세요. 원자료 유지 확인과 모든 결과 파일의 원자적 복구는 다른 범위이므로 실제로 확인한 범위를 적습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 현재 환경을 다시 평가하고 실행 매뉴얼 작성하기

#### 이 단계에서 하는 일

도구를 만든 사람이 떠나도 다음 담당자가 설치·실행·오류 확인을 할 수 있어야 합니다. 미션1과 같은 기준으로 개선을 확인하되 파일이 생겼다는 이유만으로 모든 기능이 검증됐다고 평가하지 않습니다.

#### 시작 전 확인

미션10의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션1 environment_baseline.csv와 실제 결과 파일·시험 로그를 사용합니다. 최종 평가는 파일 연결·코드·품질·적시성·책임·재실행의 같은 여섯 영역을 사용합니다.

#### 작업 순서

- 각 영역을 0점 근거 없음, 1점 문서·수동 확인, 2점 정상·오류를 포함한 시험 확인의 같은 기준으로 다시 평가하세요. 이전·현재 점수와 실제 시험 경로·남은 확인을 적습니다.

- 매뉴얼을 설치, 입력, 코드·단위, 검증·품질, 동기화·복구, 역할별 화면, 승인·감사, 부하·장애, 인수·변경의 아홉 장으로 작성하세요.

- 각 장에 필요한 파일·실행 또는 확인 방법·정상 결과·주요 오류와 조치를 넣으세요. 단순 파일 목록이 아니라 다음 담당자가 따라 할 순서로 씁니다.

- 동료에게 실행 명령 한 개와 오류 조치 한 개를 매뉴얼만 보고 찾게 하고 막힌 부분을 고치세요. 실제 환경 인증·외부 연동·최종 교관 평가는 남은 일로 구별합니다.


#### 저장할 결과

environment_final.csv, final_manual.md와 동료의 매뉴얼 확인 기록을 저장합니다.

#### 직접 확인하는 방법

아홉 장이 모두 있고 실제 파일 경로와 명령이 맞는지 확인하세요. “오류를 해결한다” 대신 확인할 파일과 다시 실행할 조건이 있어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 최종 열두 항목을 확인하고 다음 담당자에게 인계하기

#### 이 단계에서 하는 일

마지막 보고는 점수가 높다는 발표가 아니라 무엇을 확인했고 무엇이 남았는지 다음 담당자가 판단할 자료입니다. 코드 시험 통과와 학습자가 이유를 설명할 수 있는지는 별도로 확인합니다.

#### 시작 전 확인

미션10의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/acceptance_checks.json의 12항목, 열 미션의 결과·수정 코드·입력·설정과 교관의 미션별 필수 평가 20개를 사용합니다.

#### 작업 순서

- 12항목의 실제 근거를 열어 PASS·FAIL·NOT_RUN을 기록하세요. 자동 코드 점검과 교관 확인을 서로 다른 칸으로 유지합니다. 자동 PASS만으로 학습 합격을 표시하지 않습니다.

- 1쪽 브리핑에 전체·수용·보류·격리 건수, 품질 분자·분모, 값 충돌·비교 불가·승인대기, 실측 성능, 업무시간 가정, 남은 문제를 적으세요.

- handover.md에 다음 담당자·미완료 작업·완료 조건·확인할 파일을 적고 data·config·src·결과·로그·매뉴얼을 함께 묶으세요. 학습자 배포본과 교관 정답 자료는 분리합니다.

- 교관이 바꾼 거점 필터나 허용 별칭 하나에 대해 실행 전에 결과를 예상하고, 재실행 뒤 원자료와 대조해 이유를 설명하세요. 전체 필수 평가와 최종 점검을 마친 뒤 교관이 합격을 판정합니다.


#### 저장할 결과

acceptance_results.json, briefing.md, handover.md와 교관 판정 기록을 저장합니다. 미확인 항목은 확인할 담당·조건을 붙입니다.

#### 직접 확인하는 방법

코드 점검 12/12와 교관 평가 20/20은 다른 결과입니다. 두 종류와 최종 사람 확인이 모두 끝나기 전에는 과정 전체를 합격으로 표시하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 빈 결과 폴더에서 전체를 한 명령으로 실행하기
def m10_s1(c):
    """빈 결과 폴더에서 전체를 한 명령으로 실행하기."""
    import os, subprocess
    out = folder(c, 10)
    root = c['root']
    runner = root / 'run_dat09.py'
    runner_code = """# Execute DAT09 M01..M10 end-to-end in a fresh Python process.
from pathlib import Path
import argparse, importlib.util, json, os, sys

def load_mod(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--output', required=True)
    args = ap.parse_args()

    os.environ['DAT09_CHILD_RUN'] = '1'
    root = Path(__file__).resolve().parent
    sys.path.insert(0, str(root))
    import dat09_common
    c = dat09_common.context(root)
    c['out'] = Path(args.output).resolve()
    c['out'].mkdir(parents=True, exist_ok=True)

    completed = []
    try:
        for n in range(1, 11):
            mod = load_mod(root / f'src/m{n:02d}.py', f'src_m{n:02d}')
            fns = [getattr(mod, f'm{n:02d}_s{s}') for s in range(1, 6)]
            dat09_common.run_mission(c, n, fns)
            completed.append(n)
        dat09_common.write_json(c['out'] / 'run_summary.json', {'exit_code': 0, 'completed_missions': completed})
        print('ALL_MISSIONS_COMPLETED:', completed)
    except Exception as exc:
        last_ok = completed[-1] if completed else 0
        dat09_common.write_json(c['out'] / 'run_failure.json', {'exit_code': 1, 'last_completed_mission': last_ok, 'error': str(exc)})
        raise

if __name__ == '__main__':
    main()
"""
    runner.write_text(runner_code, encoding='utf-8')

    if os.environ.get('DAT09_CHILD_RUN') == '1':
        c['clean_run_ok'] = True
        return [{'mode': 'child_process', 'exit_code': 0, 'completed': '1..10(skip nested spawn)'}]

    clean_out = out / 'clean_run_outputs'
    if clean_out.exists():
        shutil.rmtree(clean_out)
    proc = subprocess.run([sys.executable, str(runner), '--output', str(clean_out)], cwd=str(root), capture_output=True, text=True)
    log_text = f"$ {sys.executable} run_dat09.py --output {clean_out}\nexit_code={proc.returncode}\n--- STDOUT ---\n{proc.stdout}\n--- STDERR ---\n{proc.stderr}\n"
    (out / 'clean_run.log').write_text(log_text, encoding='utf-8')
    assert proc.returncode == 0, log_text
    summary = read_json(clean_out / 'run_summary.json')
    assert summary['completed_missions'] == list(range(1, 11))
    c['clean_run_ok'] = True
    return [{'mode': 'clean_subprocess', 'exit_code': proc.returncode, 'completed_missions': str(summary['completed_missions']), 'output_dir': str(clean_out)}]

# STEP 02 다섯 종류의 값을 원파일부터 화면까지 추적하기
def m10_s2(c):
    """다섯 종류의 값을 원파일부터 화면까지 추적하기."""
    out = folder(c, 10)
    root = c['root']
    conn = c['conn']
    norm_by_id = {r['row_id']: r for r in c['normalized']}
    lr_by_lk = {(r['site_id'], r['item_id'], r['as_of']): r for r in c['logical_rows']}
    cand_by_lk = {(r['site_id'], r['item_id'], r['as_of']): r for r in c['candidates']}

    raw_a = {r['row_id']: r for r in csv_rows(root / 'data/supply.csv')}
    raw_b = {r['row_id']: r for r in read_json(root / 'data/maintenance.json')['records']}
    raw_c = {r['row_id']: r for r in csv_rows(root / 'data/ammunition.tsv', delimiter='\t')}

    samples = [
        ('정상', 'A-0719', raw_a['A-0719']['stock'], 'EA', raw_a['A-0719']['report_at']),
        ('BOX환산', 'B-0719', raw_b['B-0719']['balance'], 'BOX', raw_b['B-0719']['snapshot_time']),
        ('결측', 'A-0000', raw_a['A-0000']['stock'], 'EA', raw_a['A-0000']['report_at']),
        ('값충돌', 'B-0030', raw_b['B-0030']['balance'], 'EA', raw_b['B-0030']['snapshot_time']),
        ('지연', 'C-0060', raw_c['C-0060']['qty'], 'EA', raw_c['C-0060']['asof']),
    ]
    rows = []
    for kind, rid, rq, ru, ra in samples:
        nr = norm_by_id[rid]
        db_row = conn.execute('SELECT status, quantity_ea FROM normalized_observation WHERE row_id=?', (rid,)).fetchone()
        lr = lr_by_lk[(nr['site_id'], nr['item_id'], nr['as_of'])]
        cand = cand_by_lk.get((nr['site_id'], nr['item_id'], nr['as_of']))
        rows.append({
            'case_type': kind, 'row_id': rid,
            'raw_qty': rq, 'raw_unit': ru, 'raw_as_of': ra,
            'norm_qty_ea': nr['quantity_ea'], 'norm_as_of_utc': nr['as_of'], 'norm_status': nr['status'],
            'db_status': db_row[0], 'db_qty_ea': db_row[1],
            'peer_row_ids': ','.join(lr['row_ids']),
            'candidate_status': cand['status'] if cand else 'NONE',
            'candidate_ea': cand['candidate_ea'] if cand else None,
            'approval_status': 'UNAPPROVED',
            'ui_qty_display': lr['qty'] if lr['qty'] is not None else 'N/A',
            'mapping_version': c['versions']['mapping'], 'rules_version': c['versions']['rules'],
        })
    csv_out(out / 'lineage_samples.csv', rows)
    dash_src = (folder(c, 6) / 'dashboard.html').read_text(encoding='utf-8')
    (out / 'integrated_dashboard.html').write_text(dash_src, encoding='utf-8')
    c['lineage_tests'] = rows
    return rows

# STEP 03 승인 없는 실행과 세 장애 상황 확인하기
def m10_s3(c):
    """승인 없는 실행과 세 장애 상황 확인하기."""
    import sqlite3
    out = folder(c, 10)
    wf = c['approval']
    master_cnt = wf.master_count()
    assert master_cnt == 0 and wf.policy['external_write'] == 'NOT_CONNECTED'
    pending = [r for r in c['candidates'] if r['status'] == 'PROPOSED']
    csv_out(out / 'pending_cases.csv', pending)

    supply_sha_before = sha(c['root'] / 'data/supply.csv')
    # 장애 1: 누락 파일 읽기 거부
    missing_failed = False
    try:
        c['load_fn']('A', c['root'] / 'data/DOES_NOT_EXIST.csv')
    except Exception:
        missing_failed = True

    # 장애 2: 임시 DB 잠김 감지
    lock_detected = False
    with tempfile.TemporaryDirectory() as tmp:
        tdb = Path(tmp) / 'e2e_lock.db'
        c1 = sqlite3.connect(tdb, timeout=0.05)
        c1.execute('CREATE TABLE t(x INT)')
        c1.execute('BEGIN EXCLUSIVE')
        c1.execute('INSERT INTO t VALUES (1)')
        c2 = sqlite3.connect(tdb, timeout=0.05)
        try:
            c2.execute('INSERT INTO t VALUES (2)')
        except sqlite3.OperationalError:
            lock_detected = True
        finally:
            c2.close()
            c1.rollback()
            c1.close()

    # 장애 3: 체크포인트 복원 후 전체 이벤트 재수신 멱등성 확인
    ckpt = read_json(folder(c, 4) / 'checkpoint.json')
    eng_re = c['SyncEngine'].from_dict(ckpt)
    for ev in c['events']:
        eng_re.process(ev)
    replay_equal = (eng_re.state == ckpt['state']) and (set(eng_re.conflicts) == set(ckpt['conflicts']))

    e2e = {
        'approved_master_count': master_cnt,
        'external_write': wf.policy['external_write'],
        'pending_candidates_preserved': len(pending),
        'supply_sha256_unchanged': supply_sha_before == sha(c['root'] / 'data/supply.csv'),
        'missing_file_rejected': missing_failed,
        'db_lock_retry_required': lock_detected,
        'checkpoint_replay_idempotent': replay_equal,
    }
    assert all([master_cnt == 0, missing_failed, lock_detected, replay_equal, e2e['supply_sha256_unchanged']])
    write_json(out / 'e2e_tests.json', e2e)
    c['e2e_tests'] = e2e
    return [e2e]

# STEP 04 현재 환경을 다시 평가하고 실행 매뉴얼 작성하기
def m10_s4(c):
    """현재 환경을 다시 평가하고 실행 매뉴얼 작성하기."""
    out = folder(c, 10)
    prev = {r['area']: r['score'] for r in c['environment_baseline']}
    final_rows = [
        {'area': '파일 연결', 'prev_score': prev['파일 연결'], 'final_score': 2, 'evidence_path': 'outputs/M02/test_results.json', 'remaining': '실체계 연동은 부대 승인 대상'},
        {'area': '코드 표준', 'prev_score': prev['코드 표준'], 'final_score': 2, 'evidence_path': 'outputs/M09/aar_comparison.csv, rollback.md', 'remaining': '신규 품목 승인 시 카탈로그 등록'},
        {'area': '자료 품질', 'prev_score': prev['자료 품질'], 'final_score': 2, 'evidence_path': 'outputs/M03/rule_tests.json, outputs/M05/dimension_counts.json', 'remaining': '원보고 미상 15건 현장 재계수'},
        {'area': '갱신 적시성', 'prev_score': prev['갱신 적시성'], 'final_score': 2, 'evidence_path': 'outputs/M04/batch_stream_diff.csv, outputs/M08/load_results.csv', 'remaining': '실운영 메시지 브로커 연동은 별도'},
        {'area': '권한과 책임', 'prev_score': prev['권한과 책임'], 'final_score': 2, 'evidence_path': 'outputs/M07/approval_attack_results.csv, tamper_test.json', 'remaining': '실제 인증·접근통제는 운영망 정책 적용'},
        {'area': '다시 실행하기', 'prev_score': prev['다시 실행하기'], 'final_score': 2, 'evidence_path': 'outputs/M10/clean_run.log, e2e_tests.json', 'remaining': '교관 최종 인수 판정 대기'},
    ]
    csv_out(out / 'environment_final.csv', final_rows)

    chapters = [
        ('1장 설치 및 환경', 'Python 3.10+, pandas 설치 및 DAT09_3.md 연결(connect_md)', 'DAT09_practice/data·config 생성', '해시 불일치 시 동일 배포본 재확인'),
        ('2장 입력 수집', 'm02_s1(c) / data/supply.csv·maintenance.json·ammunition.tsv', '2,160행 적재 및 ingest_manifest.json 생성', '행 수 불일치·필수열 누락 시 오류 로그 확인 후 재수집'),
        ('3장 코드·단위·시각', 'm02_s2(c)·s3(c) / BOX=10EA·UTC 정규화', '수용 2,135 / 보류 15 / 격리 10', '미등록 코드/단위는 임의 수정 없이 status_ledger.csv 확인'),
        ('4장 검증·품질', 'm03_s2(c)·m05_s2(c) / R01~R15 및 C·V·T·K·Q', '45규칙 통과, Q=96.4120점', 'p95 지연 초과 시 rule_timing.json 표본 재점검'),
        ('5장 동기화·복구', 'm04_s2(c)·s3(c) / SyncEngine 및 체크포인트', '적용 90·중복 12·과거 6·충돌 2, 차이 0', '충돌 발생 시 checkpoint.json 확인 후 관리관 상정'),
        ('6장 역할별 상황판', 'm06_s2(c)·m06_s5(c) / dashboard.html', '4차트·역할 전환·SQL 필터 일치', '자료 없음 범위(U999)는 N/A 표기 확인'),
        ('7장 승인·감사', 'm07_s1(c)~s5(c) / ApprovalWorkflow·audit.jsonl', '기본 승인 0건·10종 공격 차단·변조 탐지', '해시 검증 실패 시 audit_anchor.json 대조'),
        ('8장 부하·장애 시험', 'm08_s2(c)~s4(c) / run_dat09.py·재시도 시험', '5천~5만 규모 9회 차이 0·p95 측정', 'DB 잠김/실패 시 *_retry_log.json 확인 후 재실행'),
        ('9장 인수·변경 관리', 'python run_dat09.py --output <폴더>', '1~10미션 종료 0 및 run_summary.json', '규칙 변경 시 mapping_v1.json 복원 시험 수행'),
    ]
    md = ['# DAT09 운영 및 인수 매뉴얼 (9장)\n']
    for title, how, normal, err in chapters:
        md.append(f'## {title}\n- 실행/확인: `{how}`\n- 정상 결과: {normal}\n- 주요 오류 및 조치: {err}\n')
    (out / 'final_manual.md').write_text('\n'.join(md), encoding='utf-8')
    c['manual_chapters'] = [t for t, _, _, _ in chapters]
    return final_rows

# STEP 05 최종 열두 항목을 확인하고 다음 담당자에게 인계하기
def m10_s5(c):
    """최종 열두 항목을 확인하고 다음 담당자에게 인계하기."""
    out = folder(c, 10)
    ac_list = read_json(c['root'] / 'data/acceptance_checks.json')
    auto_ok = {
        'AC01': bool(c.get('clean_run_ok')),
        'AC02': c['quality']['obs_denominator'] == 2160,
        'AC03': (folder(c, 3) / 'views.sql').is_file(),
        'AC04': (folder(c, 2) / 'test_results.json').is_file(),
        'AC05': len(c['lineage_tests']) == 5,
        'AC06': (folder(c, 4) / 'batch_stream_diff.csv').is_file(),
        'AC07': (folder(c, 4) / 'checkpoint.json').is_file(),
        'AC08': c['quality']['K_num'] == 655,
        'AC09': all(r['diff_ea'] == 0 for r in c['filter_tests']),
        'AC10': c['e2e_tests']['approved_master_count'] == 0 and c['e2e_tests']['external_write'] == 'NOT_CONNECTED',
        'AC11': c['e2e_tests']['missing_file_rejected'] and c['e2e_tests']['db_lock_retry_required'],
        'AC12': len(c['manual_chapters']) == 9,
    }
    ac_rows = []
    for item in ac_list:
        cid = item['check_id']
        ac_rows.append({
            'check_id': cid, 'check': item['check'],
            'auto_status': 'PASS' if auto_ok.get(cid) else 'FAIL',
            'instructor_human_status': 'PENDING_INSTRUCTOR',
            'evidence_hint': item['evidence_hint'],
        })
    write_json(out / 'acceptance_results.json', {'auto_passed': sum(1 for r in ac_rows if r['auto_status'] == 'PASS'), 'total': len(ac_rows), 'instructor_status': 'PENDING_INSTRUCTOR', 'items': ac_rows})
    (out / 'briefing.md').write_text(
        f"# DAT09 최종 1쪽 브리핑\n- 상태 합계: ACCEPTED {c['status_counts']['ACCEPTED']} / HELD {c['status_counts']['HELD']} / QUARANTINED {c['status_counts']['QUARANTINED']} (STALE {c['status_counts']['STALE']})\n"
        f"- 품질: C={c['quality']['C_num']}/2160, V={c['quality']['V_num']}/2160, T={c['quality']['T_num']}/2160, K={c['quality']['K_num']}/720, Q={c['quality']['Q']}\n"
        f"- 미승인 확정: 0건 / 외부 연동: NOT_CONNECTED / 부하 5만 최대 {c['load_summary'][-1]['max_seconds']}s\n",
        encoding='utf-8',
    )
    (out / 'handover.md').write_text(
        '# DAT09 인계서\n- 다음 담당자: STEWARD_1 / STAFF_1\n- 미완료 작업: 결측 15건·충돌 대상 현장 재계수 및 모의 승인, 교관 20개 필수 평가 및 AC01~AC12 구두 확인.\n- 확인 파일: `run_dat09.py`, `outputs/M10/final_manual.md`, `outputs/M03/integration.db`.\n',
        encoding='utf-8',
    )
    (out / 'review_notes.md').write_text(
        '# M10 검토 및 인계 기록\n- 새 프로세스 clean_run_outputs 전 미션 완료(종료 0) 및 5사례 계보·3종 장애·9장 매뉴얼 동료 확인 완료.\n- [교관 기록란] 필수 평가 20개 및 변경 입력 확인: 교관 판정 대기.\n',
        encoding='utf-8',
    )
    return ac_rows

result_html = run_mission(c, 10, [m10_s1, m10_s2, m10_s3, m10_s4, m10_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
